In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id=dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01.enviroment-config

In [0]:
%run ../00-common/02.bronze_helpers

In [0]:
catalog_name

In [0]:
source_file=f"{landing_folder_path}/{v_batch_id}/results"
table_name=f"{catalog_name}.{bronze_schema}.results"

In [0]:
from pyspark.sql.types import StringType, StructType, StructField, DateType, IntegerType, DoubleType

results_schema = StructType([
    StructField("date", DateType()),
    StructField("raceName", StringType()),
    StructField("round", IntegerType()),
    StructField("season", IntegerType()),
    StructField("url", StringType()),
    StructField("constructorId", StringType()),
    StructField("driverId", StringType()),
    StructField("grid", IntegerType()),
    StructField("laps", IntegerType()),
    StructField("number", IntegerType()),
    StructField("points", DoubleType()),
    StructField("position", IntegerType()),
    StructField("positionText", StringType()),
    StructField("status", StringType())
])

In [0]:
results_df=(
    spark.read
    .format('json')
    .option('mode','FAILFAST')
    .schema(results_schema)
    .load(source_file)
)

In [0]:
display(results_df)

In [0]:

results_final_df = add_ingestion_metadata(results_df)


In [0]:
display(results_final_df)

In [0]:
# (
#     results_final_df
#     .write
#     .format('delta')
#     .mode('overwrite')
#     .saveAsTable(table_name)
# )

In [0]:
write_to_bronze(
    input_df=results_final_df,
    target_table=table_name,
    batch_id=v_batch_id
)

In [0]:
display(spark.table(table_name))